In [1]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

In [2]:
PROJECT_ROOT = Path("/home/gdanzil/econ470/a0/work/hcris-data")

In [3]:
# Variable locations ------------------------------------------------------

hcris_vars = pd.DataFrame([
    ['beds', 'S300001', '01200', '0100', 'numeric'],
    ['tot_charges', 'G300000', '00100', '0100', 'numeric'],
    ['net_pat_rev', 'G300000', '00300', '0100', 'numeric'],
    ['tot_discounts', 'G300000', '00200', '0100', 'numeric'],
    ['tot_operating_exp', 'G300000', '00400', '0100', 'numeric'],
    ['ip_charges', 'G200000', '00100', '0100', 'numeric'],
    ['icu_charges', 'G200000', '01500', '0100', 'numeric'],
    ['ancillary_charges', 'G200000', '01700', '0100', 'numeric'],
    ['tot_discharges', 'S300001', '00100', '1500', 'numeric'],
    ['mcare_discharges', 'S300001', '00100', '1300', 'numeric'],
    ['mcaid_discharges', 'S300001', '00100', '1400', 'numeric'],
    ['tot_mcare_payment', 'E00A18A', '01600', '0100', 'numeric'],
    ['secondary_mcare_payment', 'E00A18A', '01700', '0100', 'numeric'],
    ['street', 'S200000', '00100', '0100', 'alpha'],
    ['city', 'S200000', '00101', '0100', 'alpha'],
    ['state', 'S200000', '00101', '0200', 'alpha'],
    ['zip', 'S200000', '00101', '0300', 'alpha'],
    ['county', 'S200000', '00101', '0400', 'alpha'],
    ['name', 'S200000', '00200', '0100', 'alpha'],
    ['uncomp_care', 'S100000', '03000', '0100', 'numeric'],
    ['cost_to_charge', 'S100000', '02400', '0100', 'numeric'],
    ['new_cap_ass', 'A700002', '00900', '0200', 'numeric'],
    ['cash', 'G000000', '00100', '0100', 'numeric'],
    ['fixed_assets', 'G000000', '02100', '0100', 'numeric'],
    ['depr_land', 'G000000', '01301', '0100', 'numeric'],
    ['depr_bldg', 'G000000', '01401', '0100', 'numeric'],
    ['depr_lease', 'G000000', '01501', '0100', 'numeric'],
    ['depr_fixed_equip', 'G000000', '01601', '0100', 'numeric'],
    ['depr_auto', 'G000000', '01701', '0100', 'numeric'],
    ['depr_major_equip', 'G000000', '01801', '0100', 'numeric'],
    ['depr_minor_equip', 'G000000', '01901', '0100', 'numeric'],
    ['current_assets', 'G000000', '01100', '0100', 'numeric'],
    ['current_liabilities', 'G000000', '03600', '0100', 'numeric'],
    ['pps_ip_charges', 'C000001', '10100', '0600', 'numeric'],
    ['pps_op_charges', 'C000001', '10100', '0700', 'numeric'],
    ['pps_mcare_cost', 'D10A181', '04900', '0100', 'numeric'],
    ['pps_pgm_cost', 'D10A181', '05300', '0100', 'numeric'],
], columns=["variable", "WKSHT_CD", "LINE_NUM", "CLMN_NUM", "source"])

# Import data -------------------------------------------------------------

year = 2008
base_path = Path("../hcris-data/HCRIS_v1996") / f"HospitalFY{year}"

hcris_alpha = pd.read_csv(
    base_path / f"hosp_{year}_ALPHA.CSV",
    header=None,
    names=['RPT_REC_NUM', 'WKSHT_CD', 'LINE_NUM', 'CLMN_NUM', 'ITM_VAL_NUM'],
    dtype=str
)

hcris_numeric = pd.read_csv(
    base_path / f"hosp_{year}_NMRC.CSV",
    header=None,
    names=['RPT_REC_NUM', 'WKSHT_CD', 'LINE_NUM', 'CLMN_NUM', 'ITM_VAL_NUM'],
    dtype=str
)

hcris_report = pd.read_csv(
    base_path / f"hosp_{year}_RPT.CSV",
    header=None,
    names=[
        'RPT_REC_NUM', 'PRVDR_CTRL_TYPE_CD', 'PRVDR_NUM', 'NPI',
        'RPT_STUS_CD', 'FY_BGN_DT', 'FY_END_DT', 'PROC_DT',
        'INITL_RPT_SW', 'LAST_RPT_SW', 'TRNSMTL_NUM', 'FI_NUM',
        'ADR_VNDR_CD', 'FI_CREAT_DT', 'UTIL_CD', 'NPR_DT',
        'SPEC_IND', 'FI_RCPT_DT'
    ],
    dtype=str
)

final_reports = hcris_report[[
    'RPT_REC_NUM', 'PRVDR_NUM', 'NPI',
    'FY_BGN_DT', 'FY_END_DT', 'PROC_DT',
    'FI_CREAT_DT', 'RPT_STUS_CD'
]].rename(columns={
    'RPT_REC_NUM': 'report',
    'PRVDR_NUM': 'provider_number',
    'NPI': 'npi',
    'FY_BGN_DT': 'fy_start',
    'FY_END_DT': 'fy_end',
    'PROC_DT': 'date_processed',
    'FI_CREAT_DT': 'date_created',
    'RPT_STUS_CD': 'status'
})

final_reports['year'] = year
final_reports['data_source'] = 'v1996'

for _, row in hcris_vars.iterrows():
    source_df = hcris_alpha if row['source'] == 'alpha' else hcris_numeric

    val = source_df[
        (source_df['WKSHT_CD'] == row['WKSHT_CD']) &
        (source_df['LINE_NUM'] == row['LINE_NUM']) &
        (source_df['CLMN_NUM'] == row['CLMN_NUM'])
    ][['RPT_REC_NUM', 'ITM_VAL_NUM']].rename(columns={
        'RPT_REC_NUM': 'report',
        'ITM_VAL_NUM': row['variable']
    })

    final_reports = final_reports.merge(val, on='report', how='left')

final_hcris_v1996 = final_reports.copy()

output_path = Path("data/output")
output_path.mkdir(parents=True, exist_ok=True)

final_hcris_v1996.to_csv(
    output_path / "HCRIS_Data_v1996.txt",
    sep='\t',
    index=False
)